In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not connected. Select Runtime → Change runtime type → GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

print("GPU:", gpu_name)
print("GPU memory:", round(gpu_memory_gb, 2), "GB")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
!nvidia-smi

Sat Aug  8 17:17:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q -U \
    transformers \
    datasets \
    accelerate \
    peft \
    bitsandbytes \
    sentencepiece \
    huggingface_hub

In [ ]:
import transformers
import datasets
import accelerate
import peft
import bitsandbytes

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

transformers: 5.14.1
datasets: 5.0.1
accelerate: 1.14.0
peft: 0.20.0
bitsandbytes: 0.50.0


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from huggingface_hub import whoami

account = whoami()
print("Logged in as:", account["name"])

Logged in as: Poojitha1997


In [ ]:
from huggingface_hub import hf_hub_download

MODEL_ID = "google/gemma-3-1b-it"

config_path = hf_hub_download(
    repo_id=MODEL_ID,
    filename="config.json",
)

print("Gemma access confirmed")
print("Config downloaded to:", config_path)

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

Gemma access confirmed
Config downloaded to: /root/.cache/huggingface/hub/models--google--gemma-3-1b-it/snapshots/dcc83ea841ab6100d6b47a070329e1ba4cf78752/config.json


In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/gemma_distillation"
CODE_DIR = f"{PROJECT_DIR}/code"
RESULTS_DIR = f"{PROJECT_DIR}/results"
LOGS_DIR = f"{PROJECT_DIR}/logs"

for folder in [CODE_DIR, RESULTS_DIR, LOGS_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Code folder:", CODE_DIR)
print("Results folder:", RESULTS_DIR)
print("Logs folder:", LOGS_DIR)

Project folder: /content/drive/MyDrive/gemma_distillation
Code folder: /content/drive/MyDrive/gemma_distillation/code
Results folder: /content/drive/MyDrive/gemma_distillation/results
Logs folder: /content/drive/MyDrive/gemma_distillation/logs


In [ ]:
import os

EVALUATOR_PATH = (
    "/content/drive/MyDrive/gemma_distillation/code/evaluate_gsm8k_v4.py"
)

if not os.path.isfile(EVALUATOR_PATH):
    raise FileNotFoundError(
        f"Evaluator not found at: {EVALUATOR_PATH}"
    )

print("Evaluator found:", EVALUATOR_PATH)
print("File size:", os.path.getsize(EVALUATOR_PATH), "bytes")

Evaluator found: /content/drive/MyDrive/gemma_distillation/code/evaluate_gsm8k_v4.py
File size: 70823 bytes


In [ ]:
!python "$EVALUATOR_PATH" --help

usage: evaluate_gsm8k_v4.py [-h] [--backend {transformers,openai}]
                            [--model MODEL] [--stage STAGE]
                            [--adapter-path ADAPTER_PATH] [--compare-base]
                            [--teacher-accuracy TEACHER_ACCURACY]
                            [--teacher-metrics-path TEACHER_METRICS_PATH]
                            [--base-url BASE_URL] [--api-key API_KEY]
                            [--timeout-seconds TIMEOUT_SECONDS]
                            [--limit LIMIT] [--max-new-tokens MAX_NEW_TOKENS]
                            [--max-input-tokens MAX_INPUT_TOKENS]
                            [--load-in-4bit]
                            [--dtype {auto,float32,float16,bfloat16}]
                            [--trust-remote-code]
                            [--disable-qwen-thinking | --no-disable-qwen-thinking]
                            [--output-dir OUTPUT_DIR] [--run-tag RUN_TAG]

Evaluate a teacher or student causal LM on GSM8K.

option

In [ ]:
import os
import subprocess
import sys

EVALUATOR_PATH = (
    "/content/drive/MyDrive/gemma_distillation/code/evaluate_gsm8k_v4.py"
)

SMOKE_RESULTS_DIR = (
    "/content/drive/MyDrive/gemma_distillation/results/"
    "gemma_before_sft_fp16_smoke"
)

os.makedirs(SMOKE_RESULTS_DIR, exist_ok=True)

command = [
    sys.executable,
    EVALUATOR_PATH,
    "--backend", "transformers",
    "--model", "google/gemma-3-1b-it",
    "--stage", "before_sft",

    "--limit", "5",

    "--max-input-tokens", "1536",
    "--max-new-tokens", "768",

    # FP16
    "--dtype", "float16",

    # IMPORTANT: no --load-in-4bit

    "--output-dir", SMOKE_RESULTS_DIR,
    "--run-tag", "fp16_smoke",
]

print("Starting 5-question Gemma FP16 smoke test...")
print("Model: google/gemma-3-1b-it")
print("Precision: FP16")
print("4-bit: OFF")

subprocess.run(command, check=True)

Starting 5-question Gemma FP16 smoke test...
Model: google/gemma-3-1b-it
Precision: FP16
4-bit: OFF


CompletedProcess(args=['/usr/bin/python3', '/content/drive/MyDrive/gemma_distillation/code/evaluate_gsm8k_v4.py', '--backend', 'transformers', '--model', 'google/gemma-3-1b-it', '--stage', 'before_sft', '--limit', '5', '--max-input-tokens', '1536', '--max-new-tokens', '768', '--dtype', 'float16', '--output-dir', '/content/drive/MyDrive/gemma_distillation/results/gemma_before_sft_fp16_smoke', '--run-tag', 'fp16_smoke'], returncode=0)

In [ ]:
# This cell runs the official Gemma BEFORE-SFT baseline on all 1,319 GSM8K questions using our fixed student protocol: FP16 with 1536 input and 768 output tokens.

import os
import subprocess
import sys

EVALUATOR_PATH = (
    "/content/drive/MyDrive/gemma_distillation/code/evaluate_gsm8k_v4.py"
)

FULL_RESULTS_DIR = (
    "/content/drive/MyDrive/gemma_distillation/results/"
    "gemma_before_sft_fp16_full"
)

os.makedirs(FULL_RESULTS_DIR, exist_ok=True)

command = [
    sys.executable,
    EVALUATOR_PATH,

    "--backend", "transformers",
    "--model", "google/gemma-3-1b-it",
    "--stage", "before_sft",

    # Fixed student evaluation token budget
    "--max-input-tokens", "1536",
    "--max-new-tokens", "768",

    # FP16 evaluation
    "--dtype", "float16",

    # No --load-in-4bit

    "--output-dir", FULL_RESULTS_DIR,
    "--run-tag", "fp16_final",
]

print("=" * 70)
print("FULL GEMMA BEFORE-SFT EVALUATION")
print("=" * 70)
print("Model: google/gemma-3-1b-it")
print("Dataset: GSM8K test")
print("Questions: 1,319")
print("Precision: FP16")
print("Max input tokens: 1536")
print("Max new tokens: 768")
print("Results directory:", FULL_RESULTS_DIR)
print("=" * 70)

subprocess.run(command, check=True)

FULL GEMMA BEFORE-SFT EVALUATION
Model: google/gemma-3-1b-it
Dataset: GSM8K test
Questions: 1,319
Precision: FP16
Max input tokens: 1536
Max new tokens: 768
Results directory: /content/drive/MyDrive/gemma_distillation/results/gemma_before_sft_fp16_full


KeyboardInterrupt: 